### Sélection et vérification des événements

#### Objectif

Ce notebook vise à vérifier la sélection de 16 extraits temporels destinés à l'expérience de sonification des données environnementales de la Senne.

L'analyse porte sur la période du **1er juillet au 30 septembre 2025** et mobilise deux types de données :

- Les précipitations mesurées par 16 pluviomètres bruxellois, avec une résolution de 5 minutes.
- Les concentrations en oxygène dissous mesurées aux stations Q01, Q02, Q03 et Q04.

L'objectif est de constituer un ensemble équilibré de **16 extraits de 2 heures** :

- 8 extraits contenant au moins un épisode de faible oxygénation.
- 8 extraits témoins sans épisode de faible oxygénation.

Un épisode de faible oxygénation est défini ici par une concentration inférieure à **3 mg/L pendant au moins 15 minutes consécutives**.

Le seuil de 3 mg/L constitue un repère environnemental (limite vie piscicole), tandis que la durée minimale de 15 minutes est un choix méthodologique destiné à écarter les fluctuations ponctuelles.

Les extraits ont été présélectionnés afin de représenter différentes situations : baisses localisées ou touchant plusieurs stations, précipitations faibles ou généralisées, et situations sans baisse d'oxygène.

#### Chargement et fusion des données

Les données pluviométriques ont été nettoyées et harmonisées lors des étapes précédentes du projet. Le fichier `pluviometres_16_fusionnes.csv` contient les mesures des 16 pluviomètres, associées à leurs horodatages.

Lors de cette préparation, les mesures prévalidées ont été privilégiées lorsqu'elles étaient disponibles et identifiées. En l'absence de telles mesures, les valeurs brutes ont été utilisées, en considérant les valeurs supérieures à 10 mm par intervalle de 5 minutes comme des valeurs d'étalonnage à exclure. Ces valeurs ont été remplacées par des valeurs manquantes (`NaN`), et non par zéro.

Les quatre fichiers d'oxygène dissous sont ensuite importés et fusionnés avec les données de précipitations à partir de la date et de l'heure.

Les fichiers d'oxygène nécessitent une lecture adaptée :

- Séparateur de colonnes : tabulation (`\t`).
- Encodage : `cp1252`.
- Séparateur décimal : virgule.
- Format des dates : jour.mois.année heure:minute:seconde.

Cette étape permet de constituer un tableau unique, nommé `df`, contenant les précipitations et l'oxygène dissous aux quatre stations.

In [ ]:

import pandas as pd
from pathlib import Path

# Dossier des données
dossier = Path("../data/totalité_juillet_septembre_2025")

# Charger les 16 pluviomètres déjà nettoyés
df = pd.read_csv(
    dossier / "pluviometres_16_fusionnes.csv",
    parse_dates=["date"]
)

# Vérifier les dates des pluviomètres
if df["date"].isna().any():
    raise ValueError("Dates invalides dans les pluviomètres")

if df["date"].duplicated().any():
    raise ValueError("Dates en double dans les pluviomètres")

# Ajouter les quatre stations d'oxygène
for station in ["Q01", "Q02", "Q03", "Q04"]:

    fichier = dossier / f"{station}.csv"

    oxygene = pd.read_csv(
        fichier,
        sep="\t",
        encoding="cp1252",
        decimal=","
    )

    # Conserver la date et l'oxygène dissous
    oxygene = oxygene.iloc[:, :2].copy()
    oxygene.columns = ["date", station]

    # Convertir les dates et les valeurs
    oxygene["date"] = pd.to_datetime(
        oxygene["date"].astype(str),
        format="%d.%m.%Y %H:%M:%S",
        errors="coerce"
    )

    oxygene[station] = pd.to_numeric(
        oxygene[station],
        errors="coerce"
    )

    # Vérifier les dates
    if oxygene["date"].isna().any():
        raise ValueError(f"Dates invalides dans {station}")

    if oxygene["date"].duplicated().any():
        raise ValueError(f"Dates en double dans {station}")

    # Fusionner sur les horodatages
    df = df.merge(
        oxygene,
        on="date",
        how="left",
        validate="one_to_one"
    )

# Conserver juillet à septembre 2025
df = df[
    (df["date"] >= "2025-07-01") &
    (df["date"] < "2025-10-01")
]

# Trier chronologiquement
df = df.sort_values("date").reset_index(drop=True)

print("Dimensions du tableau :", df.shape)
display(df.head())


Problemes apparus: 
1. L'encodage : Python ne reconnaît pas certains caractères accentués.
2. Le séparateur : Python coupe le nom date/time (SER UTC+2,00) au niveau de la virgule, alors que les colonnes des fichers d'oxygène sont séparées par des tabulations.  

Correction:  
  
   df_temp = pd.read_csv(. 
    fichier,  
    sep="\t",  
    encoding="cp1252"
) 

#### Vérification des données fusionnées

Avant d'analyser les événements, plusieurs contrôles sont réalisés :

1. Vérifier le nombre de lignes et de colonnes du tableau.
2. Compter les valeurs manquantes pour chaque variable.
3. Vérifier les valeurs minimales et maximales d'oxygène dissous.
4. Compter les mesures inférieures au seuil de 3 mg/L.

Cette dernière vérification est particulièrement importante : elle permet de détecter une éventuelle erreur d'importation des mesures d'oxygène, notamment liée au séparateur décimal.

Une valeur manquante ne doit pas être interprétée comme une absence de pluie ou comme une concentration d'oxygène nulle.

In [ ]:

# Nombre de lignes et de colonnes
print("Lignes :", len(df))
print("Colonnes :", len(df.columns))

# Valeurs manquantes
print("\nValeurs manquantes :")
print(df.isna().sum())

# Contrôle des mesures d'oxygène
print("\nContrôle de l'oxygène dissous :")

for station in ["Q01", "Q02", "Q03", "Q04"]:
    print(
        station,
        "| Minimum :", df[station].min(),
        "| Maximum :", df[station].max(),
        "| Mesures sous 3 mg/L :", (df[station] < 3).sum()
    )


#### Présélection des 16 extraits expérimentaux

À partir de l'analyse des épisodes de faible oxygénation, 16 fenêtres temporelles de deux heures ont été présélectionnées.

Huit fenêtres correspondent à des situations dans lesquelles une baisse d'oxygène est attendue. Les huit autres constituent des situations témoins.

La sélection ne repose pas uniquement sur l'intensité des baisses. Elle cherche à représenter plusieurs configurations environnementales afin d'éviter une expérience composée exclusivement d'événements très marqués, qui pourraient être trop faciles à identifier.

Les fenêtres sont ensuite vérifiées automatiquement à partir des données fusionnées. Leur catégorie initiale est conservée séparément du résultat de cette vérification, afin de pouvoir identifier les éventuelles incohérences.

In [ ]:

# Catégorie prévue et début de chaque extrait de 2 heures

extraits = [
    ("Avec baisse", "2025-07-06 12:00"),
    ("Avec baisse", "2025-07-20 17:00"),
    ("Avec baisse", "2025-07-24 19:00"),
    ("Avec baisse", "2025-08-02 14:00"),
    ("Avec baisse", "2025-08-31 17:00"),
    ("Avec baisse", "2025-09-03 22:00"),
    ("Avec baisse", "2025-09-15 07:00"),
    ("Avec baisse", "2025-09-19 21:00"),

    ("Sans baisse", "2025-07-10 12:00"),
    ("Sans baisse", "2025-07-17 06:00"),
    ("Sans baisse", "2025-07-24 12:00"),
    ("Sans baisse", "2025-08-10 12:00"),
    ("Sans baisse", "2025-08-20 12:00"),
    ("Sans baisse", "2025-08-31 06:00"),
    ("Sans baisse", "2025-09-22 12:00"),
    ("Sans baisse", "2025-09-25 12:00")
]

stations_oxygene = ["Q01", "Q02", "Q03", "Q04"]

stations_pluie = [
    c for c in df.columns
    if c not in ["date"] + stations_oxygene
]

print("Nombre d'extraits :", len(extraits))
print("Stations de pluie :", len(stations_pluie))
print("Stations d'oxygène :", len(stations_oxygene))


In [ ]:

# Vérifier les baisses d'oxygène d'au moins 15 minutes

resultats_oxygene = []

for categorie, debut in extraits:

    debut = pd.Timestamp(debut)
    fin = debut + pd.Timedelta(hours=2)

    extrait = df[
        (df["date"] >= debut) &
        (df["date"] < fin)
    ].sort_values("date")

    stations_critiques = []

    for station in stations_oxygene:

        # True lorsque l'oxygène est sous 3 mg/L
        sous_seuil = extrait[station] < 3

        # Identifier les séries de mesures consécutives
        groupes = sous_seuil.ne(sous_seuil.shift()).cumsum()

        # Durée de chaque série sous le seuil
        longueurs = sous_seuil.groupby(groupes).sum()

        # Au moins 3 mesures de 5 minutes = 15 minutes
        if (longueurs >= 3).any():
            stations_critiques.append(station)

    resultats_oxygene.append({
        "Début": debut,
        "Catégorie prévue": categorie,
        "Épisode ≥ 15 min": (
            "OUI" if stations_critiques else "NON"
        ),
        "Stations concernées": (
            ", ".join(stations_critiques)
            if stations_critiques else "Aucune"
        )
    })

verification_oxygene = pd.DataFrame(resultats_oxygene)

display(verification_oxygene)


#### Analyse et caractérisation des précipitations

Pour chaque fenêtre de deux heures, les précipitations sont analysées à partir des 16 pluviomètres.

Deux indicateurs sont calculés :

- Le nombre de pluviomètres ayant enregistré une précipitation strictement positive pendant la fenêtre.
- Le cumul maximal de précipitations sur deux heures, parmi les pluviomètres.

Ces indicateurs permettent de distinguer les situations sans pluie enregistrée, les précipitations localisées et les précipitations plus étendues.

Le cumul maximal correspond à la valeur d'un seul pluviomètre : il ne représente ni la moyenne spatiale ni le cumul de l'ensemble des 16 stations.

L'analyse porte ici sur les précipitations pendant la fenêtre étudiée. Elle ne permet pas, à elle seule, de caractériser les pluies survenues avant la baisse d'oxygène.

#### Verification de la pluie

Étape intermédiaire qui m'as permis de distinguer - entre autre - les précipitations généralisées du 6 juillet et les précipitations du 31 août au matin.

In [ ]:

resultats_pluie = []

for categorie, date in extraits:

    debut = pd.Timestamp(date)
    fin = debut + pd.Timedelta(hours=2)

    extrait = df[
        (df["date"] >= debut) &
        (df["date"] < fin)
    ]

    # Cumul de pluie pour chaque pluviomètre
    cumuls = extrait[stations_pluie].sum(min_count=1)

    resultats_pluie.append({
        "Début": debut,
        "Catégorie prévue": categorie,
        "Mesures": len(extrait),
        "Pluviomètres avec pluie": int((cumuls > 0).sum()),
        "Pluie maximale (mm)": cumuls.max()
    })

verification_pluie = pd.DataFrame(resultats_pluie)

display(verification_pluie)


#### Vérification des épisodes de faible oxygénation

La vérification des événements repose sur deux critères :

- Une concentration en oxygène dissous strictement inférieure à 3 mg/L.
- Le maintien de cette condition pendant au moins trois mesures consécutives espacées de cinq minutes, correspondant à une durée nominale de 15 minutes.

Les séquences sont recherchées indépendamment pour les quatre stations Q01, Q02, Q03 et Q04.

Les mesures manquantes et les interruptions temporelles ne sont pas considérées comme des mesures consécutives.

Pour chaque extrait, le programme identifie les stations présentant un épisode répondant à ces critères. Il compare ensuite le résultat à la catégorie prévue lors de la présélection.

In [ ]:

resultats_oxygene = []

for categorie, date in extraits:

    debut = pd.Timestamp(date)
    fin = debut + pd.Timedelta(hours=2)

    extrait = df[
        (df["date"] >= debut) &
        (df["date"] < fin)
    ].sort_values("date").copy()

    stations_critiques = []

    for station in stations_oxygene:

        # Mesures sous le seuil
        sous_seuil = extrait[station] < 3

        # Commencer un nouveau groupe si :
        # - le statut sous le seuil change
        # - les mesures ne sont pas espacées de 5 minutes
        nouveau_groupe = (
            sous_seuil.ne(sous_seuil.shift()) |
            extrait["date"].diff().ne(pd.Timedelta(minutes=5))
        )

        groupes = nouveau_groupe.cumsum()

        # Nombre de mesures consécutives sous le seuil
        longueurs = sous_seuil.groupby(groupes).sum()

        if (longueurs >= 3).any():
            stations_critiques.append(station)

    episode = len(stations_critiques) > 0

    # Comparer le résultat à la catégorie prévue
    attendu = categorie == "Avec baisse"

    resultats_oxygene.append({
        "Début": debut,
        "Catégorie prévue": categorie,
        "Épisode ≥ 15 min": "OUI" if episode else "NON",
        "Stations concernées": (
            ", ".join(stations_critiques)
            if episode else "Aucune"
        ),
        "Conforme": "OUI" if episode == attendu else "NON",
        "Mesures": len(extrait)
    })

verification_oxygene = pd.DataFrame(resultats_oxygene)

display(verification_oxygene)


#### Bilan finale

In [ ]:

# Compter les extraits conformes à leur catégorie

print("BILAN DE LA SÉLECTION")
print("---------------------")

print("Extraits analysés :", len(verification_oxygene))

print(
    "Extraits conformes :",
    (verification_oxygene["Conforme"] == "OUI").sum()
)

print(
    "Extraits à revoir :",
    (verification_oxygene["Conforme"] == "NON").sum()
)

print(
    "Extraits avec 24 mesures :",
    (verification_oxygene["Mesures"] == 24).sum()
)

# Afficher les extraits qui nécessitent une vérification
a_revoir = verification_oxygene[
    (verification_oxygene["Conforme"] == "NON") |
    (verification_oxygene["Mesures"] != 24)
]

if a_revoir.empty:
    print("\nAucune incohérence détectée.")
else:
    print("\nExtraits à examiner :")
    display(a_revoir)


Pour avoir le cvs qui ressamble les evenements choisi

In [ ]:
# Enregistrer les 16 événements validés
from pathlib import Path

selection = pd.DataFrame(
    extraits,
    columns=["categorie", "debut"]
)

selection["debut"] = pd.to_datetime(selection["debut"])

# Vérifier que la sélection est conforme
assert len(selection) == 16
assert (verification_oxygene["Conforme"] == "OUI").all()
assert (verification_oxygene["Mesures"] == 24).all()

fichier = Path("../data/selection_stimuli.csv")
selection.to_csv(fichier, index=False)

print("Sélection enregistrée :", fichier)

#### Bilan et limites de la sélection

La vérification automatisée permet de contrôler la présence ou l'absence d'épisodes de faible oxygénation dans les 16 fenêtres présélectionnées.

La sélection finale vise à maintenir un équilibre entre les extraits contenant un événement et les situations témoins, tout en conservant une diversité de conditions pluviométriques et de configurations spatiales.

Plusieurs limites doivent être prises en compte :

- Le seuil de 3 mg/L et la durée minimale de 15 minutes correspondent à deux choix de nature différente : le premier est un repère environnemental, tandis que le second constitue un critère méthodologique.
- La présence de précipitations pendant un extrait ne démontre pas qu'elles sont responsables de la baisse d'oxygène observée.
- Des précipitations antérieures à la fenêtre étudiée peuvent également intervenir.
- Les mesures manquantes limitent la possibilité de confirmer l'absence d'un épisode.
- La sélection raisonnée de 16 extraits permet de construire un protocole expérimental diversifié, mais ne constitue pas un échantillonnage aléatoire de l'ensemble des conditions environnementales.

Ces résultats serviront à préparer les extraits audiovisuels de l'expérience de sonification.